In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#Importing Libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr
import os; import time
import pickle
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
import h5py

import sys
from tqdm import tqdm

In [ ]:
#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

In [ ]:
####################################
#LOADING CLASSES

In [ ]:
#IMPORT CLASSES (from current directory)
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

In [ ]:
#IMPORT FUNCTIONS
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
import FUNCTIONS_Variable_Calculation
from FUNCTIONS_Variable_Calculation import *

In [ ]:
#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=1)
#data manager class
DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="LagrangianArrays", dataName="PROCESSED_Lagrangian_Binary_Array",
                                dtype='int8')

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","2_Tracking_Algorithms"))
from CLASSES_TrackingAlgorithms import SlurmJobArray_Class

In [ ]:
import sys
path=os.path.join(mainCodeDirectory,'Functions/')
sys.path.append(path)

import NumericalFunctions
from NumericalFunctions import * # import NumericalFunctions 
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions


# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(PlottingFunctions, inspect.isfunction)]
# functions

In [ ]:
##############################
#SETUP

In [ ]:
################################
#JOB ARRAY SETUP
################################
#*#*
# how many total jobs are being run? i.e. array=1-100 ==> num_jobs=100
if ModelData.Np_str=='1e6': #1M parcels
    num_jobs=60 
    num_slurm_jobs=20 #this is the number that goes into sbatch script (note: this is opposite of LagrangianUpdraftTracking by accident)
elif ModelData.Np_str=='20e6': #1M parcels
    num_jobs=100 
    num_slurm_jobs=30 #this is the number that goes into sbatch script
elif ModelData.Np_str=='50e6': #50M parcels
    num_jobs=300 
    num_slurm_jobs=100 #this is the number that goes into sbatch script
##############################

In [ ]:
##############################################
#DATA LOADING FUNCTIONS

In [ ]:
#Initialize Data Functions
def InitiateArray(f, vars, t_chunk_size, p_chunk_size, t_size=None, p_size=None, Np=None):
    if t_size is None:
        t_size = ModelData.Ntime  # Number of timesteps
    if p_size is None:
        p_size = Np if Np is not None else ModelData.Np  # Number of parcels (job-sized or full)

    for var_name in vars:
        if var_name not in f:
            # Set dtype conditionally
            if var_name in ['Z', 'Y', 'X']:
                dtype = np.uint16
            elif var_name in ['A_g','A_c','PROCESSED_A_g','PROCESSED_A_c']:
                dtype = np.bool_
            else:
                dtype = np.float32  # or whatever your default is
            f.create_dataset(
                var_name,
                shape=(t_size, p_size),
                chunks=(t_chunk_size, p_chunk_size),
                dtype=dtype
            )

In [ ]:
# Loading Data Functions
def Get_Lagrangian_Binary_Array_Data_V1(ModelData, start_job,end_job):
    directory_Lagrangian_Binary_Array = f"/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/Code/OUTPUT/Variable_Calculation/LagrangianArrays/Simulation_{ModelData.simulationNumber}_{ModelData.res}_{ModelData.t_res}_{ModelData.Nz_str}nz/Lagrangian_Binary_Array/"
    Lagrangian_Binary_Array_Data,files = OpenMultipleSingleTimes_LagrangianArray_JobArray(directory_Lagrangian_Binary_Array, ModelData, start_job,end_job)
    return Lagrangian_Binary_Array_Data

def GetArrays_V1(Lagrangian_Binary_Array_Data):    
    # 1. Subset the dataset to just the two variables and compute in one pass
    ds = Lagrangian_Binary_Array_Data[['A_g', 'A_c']].compute()
    
    # 2. Extract the raw NumPy arrays
    A_g = ds['A_g'].data
    A_c = ds['A_c'].data
    
    return A_g, A_c

import glob as glob
def Get_Lagrangian_Binary_Array_Data_V2(ModelData, start_job, end_job, varNames=['A_g','A_c'],
                                       pattern="Lagrangian_Binary_Array_*.h5"):
    """
    Loads a parcel slice [start_job:end_job] across all timestep files directly
    into preallocated NumPy arrays. Simple h5py replacement for the xarray loader.
    Returns arrays shaped (Ntime, Np_job), matching GetArrays' output.

    This version is much faster than V1, however V1 may win for much larger data.
    """
    directory = (f"/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/"
                 f"Code/OUTPUT/Variable_Calculation/LagrangianArrays/"
                 f"Simulation_{ModelData.simulationNumber}_{ModelData.res}_"
                 f"{ModelData.t_res}_{ModelData.Nz_str}nz/Lagrangian_Binary_Array/")

    #--- glob once, then match each timestep by suffix (same strategy as the xarray loader)
    filesAll =  glob.glob(os.path.join(directory, pattern))
    if not filesAll:
        raise FileNotFoundError(f"No files found in {directory} matching {pattern}")

    files = []
    for t in ModelData.timeStrings:
        matched = [f for f in filesAll if f.endswith(f"_{t}.h5")]
        if not matched:
            raise FileNotFoundError(f"Missing file for time {t}")   #hard stop, not silent skip
        files.append(matched[0])

    Np_job = end_job - start_job
    arrays = {v: np.empty((ModelData.Ntime, Np_job), dtype=bool) for v in varNames}

    for i, file in enumerate(tqdm(files, desc=f"Loading time files for p in ({start_job},{end_job})")):
        with h5py.File(file, "r") as fin:
            for v in varNames:
                arrays[v][i] = fin[v][start_job:end_job]   #one contiguous hyperslab per variable

    return [arrays[v] for v in varNames]

def GetArrays_V2(Lagrangian_Binary_Array_Data):
    [A_g, A_c] = Lagrangian_Binary_Array_Data
    return [A_g, A_c]

In [ ]:
################################
#FUNCTIONS

In [ ]:
#Helper Functions

def extend_idxs(f,case):
    out=np.sort(np.add.outer(f, np.arange(case)).ravel())

    # #OLD METHOD (SLOW)
    # if np.any(f)==True:
    #     out=np.sort(np.concatenate([np.arange(idx, idx + case-1+1) for idx in f]))
    # else: 
    #     out=f
    return out

def find_sandwiched_patterns(changes, case):
    arr=changes
    
    window_size = case + 1  # e.g., for case=2, window_size = 3
    # The interior zeros count is (window_size - 2) which is case - 1
    pattern1 = np.array([-1] + [0]*(case - 1) + [1])
    pattern2 = np.array([1] + [0]*(case - 1) + [-1])
    # print(pattern1,pattern2)
    
    # Manually construct sliding windows
    windows = np.array([arr[i:i + window_size] for i in range(len(arr) - window_size + 1)])
    # print("Sliding windows:\n", windows) #TESTING
    
    #THE ALGORITHM
    turb_d=[]
    turb_e=[]
    count=0;max_iter=ModelData.Ntime;
    while np.any(((windows == pattern1) | (windows == pattern2)).all(axis=1)):
        count+=1; 
        if count>=max_iter: 
            print(count)
            break
        
        next_ind = np.where(((windows == pattern1) | (windows == pattern2)).all(axis=1))[0][0]
        
        if (windows[next_ind] == pattern1).all():
            turb_d.append(next_ind)
        elif (windows[next_ind] == pattern2).all(): 
            turb_e.append(next_ind) #append to list
    
        windows[0:next_ind+(case)+1,:] = 0 #removes from windows
    
    turb_d=np.array(turb_d,dtype=int); turb_e=np.array(turb_e,dtype=int)

    #EXTEND REST OF INDEXES TO PROCESS
    turb_d=extend_idxs(turb_d,case=case)
    turb_e=extend_idxs(turb_e,case=case)
    return turb_d,turb_e

In [ ]:
#Calculation Functions

###### (amount of time inside/outside of cloud to count as entrainment/detrainment)
mins_thresh=5 #5 mins
######

t_per_mins=1/((ModelData.time[1]-ModelData.time[0])/1e9/60).item() #timesteps per minute (<=1)
def get_changes(B):
    changes = np.diff(np.concatenate(([B[0]], B)))  # Add 0s to detect edges
    return changes
def PreProcessing(A, p, B=None, case=None, testing=False):
    if B is None:
        B = A[:,p]*1
    else:
        B = B.copy()   #don't mutate the caller's test array in place

    #Determining the Case Number
    if case is None:
        case = int(t_per_mins*mins_thresh)
    if testing:
        TestPrint("case", case)
        
    # Find the changes in the array
    changes = get_changes(B)
    if testing:
        TestPrint()
        TestPrint("B (start)", B)
        TestPrint("changes", changes)

    if case > 1:
        for case_ind in np.arange(case, 0, -1):
            [turb_d, turb_e] = find_sandwiched_patterns(changes, case=case_ind)
            B[turb_d] = 1
            B[turb_e] = 0
            changes = get_changes(B)
            if testing:
                TestPrint()
                TestPrint(f"case_ind={case_ind}: turb_d", turb_d)
                TestPrint(f"case_ind={case_ind}: turb_e", turb_e)
                TestPrint(f"case_ind={case_ind}: B", B)
    elif case == 1:
        [turb_d, turb_e] = find_sandwiched_patterns(changes, case=case)
        B[turb_d] = 1
        B[turb_e] = 0
        if testing:
            TestPrint()
            TestPrint("case=1: turb_d", turb_d)
            TestPrint("case=1: turb_e", turb_e)
            TestPrint("B_out", B)

    return [B]

def TestPrint(name="", array=None):
    if array is None:
        print(name)
    else:
        print(f'{name} =\n', array)

In [ ]:
#Combined Running Functions
def Apply(A_g,A_c, Np):
    """
    Removes 5-minute Turbulent Entrainment
    Note: should only be used for entrainment calculations 
    and not for locating Eulerian updrafts
    """
    PROCESSED_A_g = A_g.copy()
    PROCESSED_A_c = A_c.copy()
    for p in tqdm(np.arange(Np)):
        out1=PreProcessing(PROCESSED_A_g,p); PROCESSED_A_g[:,p]=out1
        out2=PreProcessing(PROCESSED_A_c,p); PROCESSED_A_c[:,p]=out2
    return PROCESSED_A_g,PROCESSED_A_c

def CheckDifferences(A_g,A_c, PROCESSED_A_g,PROCESSED_A_c):
    diff_g = np.sum(A_g != PROCESSED_A_g)
    diff_c = np.sum(A_c != PROCESSED_A_c)
    print(f"Differences: A_g={diff_g}, A_c={diff_c}")

#SAVING
def Save(Np,PROCESSED_A_g,PROCESSED_A_c,job_id):
    out_file = (
        DataManager.outputDataDirectory
        + f'/PROCESSED_Lagrangian_Binary_Array_{ModelData.res}_'
        f'{ModelData.t_res}_{ModelData.Nz_str}nz_{job_id}.h5'
    )
    print(f"Saving as {out_file}","\n")
    
    varNames = ['PROCESSED_A_g', 'PROCESSED_A_c']
    with h5py.File(out_file, 'w') as f:
        InitiateArray(f, varNames, t_chunk_size=50, p_chunk_size=Np, p_size=Np)
        f['PROCESSED_A_g'][:] = PROCESSED_A_g
        f['PROCESSED_A_c'][:] = PROCESSED_A_c

def RunCode(job_id_list):
    for job_id in job_id_list:
        if job_id % 1 ==0: print(f"current job_id = {job_id}\n")
        [start_job,end_job,index_adjust]=SlurmJobArray_Class.StartJobArray(ModelData, job_id, num_jobs)
        print(f'Running on Parcels {start_job}-{end_job}')
        Np=end_job-start_job
    
        Lagrangian_Binary_Array_Data=Get_Lagrangian_Binary_Array_Data_V2(ModelData, start_job,end_job)
        [A_g,A_c]=GetArrays_V2(Lagrangian_Binary_Array_Data)
    
        [PROCESSED_A_g,PROCESSED_A_c]=Apply(A_g,A_c, Np)
        CheckDifferences(A_g,A_c, PROCESSED_A_g,PROCESSED_A_c)
        Save(Np,PROCESSED_A_g,PROCESSED_A_c,job_id)
    return PROCESSED_A_g,PROCESSED_A_c

In [ ]:
####################################
#RUNNING
running=True #KEEP TRUE WHEN JOB ARRAY IS RUNNING
# running=False 

In [ ]:
if running:
    #Running
    [start_slurm_job,end_slurm_job]=SlurmJobArray_Class.StartSlurmJobArray(num_jobs=num_jobs,num_slurm_jobs=num_slurm_jobs,ISRUN=True) #if ISRUN is False, then will not run using slurm_job_array
    
    print(f"Running on Slurm_Jobs for Slurm_Job_Ids: {(start_slurm_job,end_slurm_job-1)}")
    
    job_id_list=np.arange(start_slurm_job,end_slurm_job)
    
    [PROCESSED_A_g,PROCESSED_A_c] = RunCode(job_id_list)

In [ ]:
#COMBINING JOB_ARRAYS AFTER RUNNING
########################################################################

In [ ]:
#Job Array Functions
import glob as glob
import re, os
#IMPORT CLASSES (from current directory)
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class

#JOB ARRAY SETUP
UsingJobArray=True #*if taking too long set to true and run with slurm

def GetNumJobs(res,t_res):
    if res=='1km':
        if t_res=='5min':
            num_jobs=20
        elif t_res=='3min': #1M parcels
            num_jobs=30
        elif t_res=='1min':
            num_jobs=100
    return num_jobs
num_jobs = GetNumJobs(ModelData.res,ModelData.t_res)
SlurmJobArray = SlurmJobArray_Class(total_elements=ModelData.Ntime, num_jobs=num_jobs, UsingJobArray=UsingJobArray)
start_job = SlurmJobArray.start_job; end_job = SlurmJobArray.end_job

def GetNumElements():
    loop_elements = np.arange(ModelData.Ntime)[start_job:end_job]
    return loop_elements

In [ ]:
#Recombining Functions
def Recombine_ByTime_V1(testing=False):
    """
    This one uses job array.
    """

    loop_elements = GetNumElements()

    varNames = ['PROCESSED_A_g', 'PROCESSED_A_c']

    in_files = (DataManager.outputDataDirectory
        + f'/PROCESSED_Lagrangian_Binary_Array_{ModelData.res}_'
          f'{ModelData.t_res}_{ModelData.Nz_str}nz_*.h5')

    matching_files = sorted(
        (f for f in glob.glob(in_files)
         if re.search(r'_(\d+)\.h5$', os.path.basename(f))),
        key=lambda f: int(re.search(r'_(\d+)\.h5$', os.path.basename(f)).group(1))
    )

    outDir = DataManager.outputDataDirectory
    os.makedirs(outDir, exist_ok=True)

    # ---- compute parcel counts ONCE
    parcelCounts = []
    for f in matching_files:
        with h5py.File(f, "r") as fin:
            parcelCounts.append(fin['PROCESSED_A_g'].shape[1])

    totalParcels = sum(parcelCounts)
    nVar = len(varNames)

    # ---- loop over time
    for t in tqdm(loop_elements, desc="Recombining time"):
        stacked = np.empty((nVar, totalParcels), dtype=bool)

        offset = 0
        for f, nParcel in tqdm(zip(matching_files, parcelCounts),
                               total=len(matching_files),leave=False):
            with h5py.File(f, "r") as fin:
                stacked[0, offset:offset + nParcel] = fin['PROCESSED_A_g'][t]
                stacked[1, offset:offset + nParcel] = fin['PROCESSED_A_c'][t]
                offset += nParcel

        if testing:
            return [stacked[0].copy(), stacked[1].copy(), t]         

        outFile = (
            f"{outDir}/PROCESSED_Lagrangian_Binary_Array_"
            f"{ModelData.res}_{ModelData.t_res}_{ModelData.Nz_str}nz_"
            f"{ModelData.timeStrings[t]}.h5"
        )

        with h5py.File(outFile, "w", libver="latest") as fout:
            fout.create_dataset(
                "PROCESSED_A_g", data=stacked[0], dtype="bool",
                chunks=(totalParcels,), compression=None
            )
            fout.create_dataset(
                "PROCESSED_A_c",
                data=stacked[1], dtype="bool",
                chunks=(totalParcels,),compression=None
            )
        print(f"Saved to {outFile}")
    return None

def Recombine_ByTime_V2(testing=False):
    """
    This one uses job array.
    """
    loop_elements = GetNumElements()
    varNames = ['PROCESSED_A_g', 'PROCESSED_A_c']
    in_files = (DataManager.outputDataDirectory
        + f'/PROCESSED_Lagrangian_Binary_Array_{ModelData.res}_'
          f'{ModelData.t_res}_{ModelData.Nz_str}nz_*.h5')
    matching_files = sorted(
        (f for f in glob.glob(in_files)
         if re.search(r'_(\d+)\.h5$', os.path.basename(f))),
        key=lambda f: int(re.search(r'_(\d+)\.h5$', os.path.basename(f)).group(1))
    )
    outDir = DataManager.outputDataDirectory
    os.makedirs(outDir, exist_ok=True)
    parcelCounts = []
    for f in matching_files:
        with h5py.File(f, "r") as fin:
            parcelCounts.append(fin['PROCESSED_A_g'].shape[1])
    totalParcels = sum(parcelCounts)

    tStart, tEnd = loop_elements[0], loop_elements[-1] + 1
    nT = tEnd - tStart

    blocksG, blocksC = [], []
    for f in tqdm(matching_files, desc="Reading job files", leave=False):
        with h5py.File(f, "r") as fin:
            blocksG.append(fin['PROCESSED_A_g'][tStart:tEnd])
            blocksC.append(fin['PROCESSED_A_c'][tStart:tEnd])
    allG = np.concatenate(blocksG, axis=1)
    allC = np.concatenate(blocksC, axis=1)

    pbar = tqdm(loop_elements, desc="Recombining time")   #<<< keep a handle on the bar
    for t in pbar:
        tLocal = t - tStart
        pbar.set_postfix({"t": t})   #<<< show which timestep is being saved

        if testing:
            return [allG[tLocal].copy(), allC[tLocal].copy(), t]

        outFile = (
            f"{outDir}/PROCESSED_Lagrangian_Binary_Array_"
            f"{ModelData.res}_{ModelData.t_res}_{ModelData.Nz_str}nz_"
            f"{ModelData.timeStrings[t]}.h5"
        )
        with h5py.File(outFile, "w", libver="latest") as fout:
            fout.create_dataset("PROCESSED_A_g", data=allG[tLocal], dtype="bool",
                                 chunks=(totalParcels,), compression=None)
            fout.create_dataset("PROCESSED_A_c", data=allC[tLocal], dtype="bool",
                                 chunks=(totalParcels,), compression=None)
        print(f"Saved to {outFile}")
    return None

def Recombine_ByTime_V3(batchSize=50, testing=False, testT=100):
    """
    This one does not use job_array.
    """
    varNames = ['PROCESSED_A_g', 'PROCESSED_A_c']
    in_files = (DataManager.outputDataDirectory
        + f'/PROCESSED_Lagrangian_Binary_Array_{ModelData.res}_'
          f'{ModelData.t_res}_{ModelData.Nz_str}nz_*.h5')
    matching_files = sorted(
        (f for f in glob.glob(in_files)
         if re.search(r'_(\d+)\.h5$', os.path.basename(f))),
        key=lambda f: int(re.search(r'_(\d+)\.h5$', os.path.basename(f)).group(1))
    )
    outDir = DataManager.outputDataDirectory
    os.makedirs(outDir, exist_ok=True)

    fileHandles = [h5py.File(f, "r") for f in matching_files]
    parcelCounts = [fh['PROCESSED_A_g'].shape[1] for fh in fileHandles]
    totalParcels = sum(parcelCounts)

    pbar = tqdm(total=ModelData.Ntime, desc="Recombining time")   #<<< one bar, all timesteps

    try:
        for bStart in range(0, ModelData.Ntime, batchSize):
            bEnd = min(bStart + batchSize, ModelData.Ntime)

            blocksG = [fh['PROCESSED_A_g'][bStart:bEnd] for fh in fileHandles]
            blocksC = [fh['PROCESSED_A_c'][bStart:bEnd] for fh in fileHandles]
            allG = np.concatenate(blocksG, axis=1)
            allC = np.concatenate(blocksC, axis=1)

            for t in range(bStart, bEnd):
                tLocal = t - bStart
                pbar.set_postfix({"t": t})   #<<< show which timestep is being saved
                pbar.update(1)               #<<< advance the bar by one, right here

                if testing:
                    if t != testT: continue
                    return [allG[tLocal].copy(), allC[tLocal].copy(), t]

                outFile = (
                    f"{outDir}/PROCESSED_Lagrangian_Binary_Array_"
                    f"{ModelData.res}_{ModelData.t_res}_{ModelData.Nz_str}nz_"
                    f"{ModelData.timeStrings[t]}.h5"
                )
                with h5py.File(outFile, "w", libver="latest") as fout:
                    fout.create_dataset("PROCESSED_A_g", data=allG[tLocal], dtype="bool",
                                         chunks=(totalParcels,), compression=None)
                    fout.create_dataset("PROCESSED_A_c", data=allC[tLocal], dtype="bool",
                                         chunks=(totalParcels,), compression=None)
    finally:
        for fh in fileHandles:
            fh.close()
        pbar.close()   #<<< always close the bar, even on early return or error

    return None

# [g1, c1, t1] = Recombine_ByTime_V1(testing=True)
# [g2, c2, t2] = Recombine_ByTime_V2(testing=True)
# [g3, c3, t3] = Recombine_ByTime_V3(testing=True)

# print(f"t1={t1}, t2={t2}, t3={t3}")

# print("A_g: V1 vs V2:", np.array_equal(g1, g2))
# print("A_g: V1 vs V3:", np.array_equal(g1, g3))
# print("A_g: V2 vs V3:", np.array_equal(g2, g3))

# print("A_c: V1 vs V2:", np.array_equal(c1, c2))
# print("A_c: V1 vs V3:", np.array_equal(c1, c3))
# print("A_c: V2 vs V3:", np.array_equal(c2, c3))

In [ ]:
if not running:
    Recombine_ByTime_V3()

In [ ]:
####################################
#TESTING

In [ ]:
# #testing #Test Cases
# testCases={ #for case=2
#     #--- sample test case ---
#     1: np.array([1,1,1,1,0,0,0,1,1,1,1,0,0,0,1,0,0,1,1,1,1,0,0,1]),   #expected: [1,1,1,1,0,0,0,1,1,1,1,0,0,0,0,0,0,1,1,1,1,1,1,1]
    
#     #--- trivial / degenerate cases (nRuns < 3, must return unchanged) ---
#     2:  np.array([1,1,1,1,1,1,1,1]),                                    #constant in-cloud: no change
#     3:  np.array([0,0,0,0,0,0,0,0]),                                    #constant out-of-cloud: no change
#     4:  np.array([0,0,0,0,1,1,1,1]),                                    #single transition (2 boundary runs): no change
#     5:  np.array([0,0,0,0,0,0,0,1]),                                    #single transition, short last run: STILL no change (boundary)

#     #--- isolated short excursions (both directions) ---
#     6:  np.array([1,1,1,1,0,1,1,1,1]),                                  #1-step dropout: filled -> all 1s
#     7:  np.array([1,1,1,1,0,0,1,1,1]),                                  #2-step dropout (=case): filled -> all 1s
#     8:  np.array([1,1,1,1,0,0,0,1,1]),                                  #3-step dropout (=case+1): SURVIVES unchanged
#     9:  np.array([0,0,0,0,1,1,0,0,0]),                                  #2-step blip INTO cloud: filled -> all 0s
#     10:  np.array([0,0,0,1,1,1,0,0,0]),                                  #3-step entrainment: survives unchanged

#     #--- boundary runs (never modified, even when short) --- #*
#     11: np.array([1,0,0,0,1,1,1,1,1]),                                  #short FIRST run: index 0 stays 1
#     12: np.array([1,1,1,1,1,0,0,0,1]),                                  #short LAST run: index 8 stays 1
#     13: np.array([1,0,0,0,0,0,0,0,1]),                                  #short runs at BOTH ends: unchanged everywhere

#     #--- clusters: the divergence regime (adjacent short runs) ---
#     14: np.array([1,0,1,1,0,1]),                                        #minimal divergent case: -> all 1s (countdown would give 1,0,0,0,0,1)
#     15: np.array([0,0,0,1,1,0,0,1,1,1,1,1,1,0,0,0]),                    #flicker prelude before real entrainment
#     16: np.array([1,1,1,0,0,1,1,0,1,1,0,0,1,1,1]),                      #long flicker chain, 1-anchors both sides: -> all 1s
#     17: np.array([0,1,0,1,0,1,0,1,0,0,0]),                              #pure alternation with 0 boundary anchor: -> all 0s

#     #--- multiple anchors with DIFFERENT values (independent local fills) ---
#     18: np.array([0,0,0,1,0,0,1,1,1,0,1,1,0,0,0]),                      #short runs fill toward their own local anchor
#     19: np.array([1,1,1,0,0,0,1,1,0,0,0,1,1,1,0,0,0]),                  #alternating anchors, 2-step blips absorb into LEFT anchor

#     #--- edge sizes ---
#     20: np.array([1,0,1]),                                              #3 elements, short interior run: -> all 1s
#     21: np.array([1,0]),                                                #2 elements, nRuns=2: unchanged
#     22: np.array([1]),                                                  #1 element: unchanged
#     23: np.array([0,1,1,0]),                                            #interior 2-run flanked by short boundary anchors: -> all 0s

#     #--- testing boundary case (14) effect on rest of run ---
#     24: np.array([1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1])
# }

# B = testCases[24] #only case 1 doesn't match the new version using maximum.accumulate
# [B_out] = PreProcessing(A=None,p=0,B=B,case=2,testing=True)